# Sketch 2.6 - standalone, portable notebook

Self-contained extract of **sketch_2_6_small_multiples** from the combined *9 sketches*
notebook. The sketch code is unchanged; the shared setup it needs
(imports, data loading, department outlines) is included below. Chart data is **inlined** (no external
`altair-data` files) so it renders on Linux, macOS and Windows alike.

The searchable name universe is restricted to the top-40 names by total (plus the
sketch's defaults) so the inlined data stays small; the default view is
unchanged.

## Shared setup *(unchanged)*

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF-CONTAINED and renders on
# all platforms (incl. macOS): external altair-data-*.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches (1.13 / 2.4 / 2.6) restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [ ]:
# Memory-lean load: another heavy job is running on this machine (~2.5 GB free),
# so we read the CSV with categorical dtypes (15k name strings stored once), derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects (the strings stay shared) so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all-years aggregate (2.4).
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


## Name-universe restriction *(portability/memory)*

In [ ]:
# Portability + memory: keep the INLINED data small by limiting the name
# universe to names that are in the top 40 by total births, always unioned with this sketch's own
# default names so the default view is unchanged.
import gc
_EXTRA = [x for x in ['LUCIEN', 'LEA', 'GABRIEL', 'NICOLAS', 'EMMA', 'LOUIS'] if x in set(named['preusuel'])]
_universe = set(named.groupby('preusuel')['nombre'].sum().nlargest(40).index)
POPULAR_NAMES = sorted(_universe.union(_EXTRA))
named = named[named['preusuel'].isin(POPULAR_NAMES)].copy()
ALL_NAMES = POPULAR_NAMES
gc.collect()
print(f'Universe: {len(POPULAR_NAMES):,} names (are in the top 40 by total births + defaults); {len(named):,} rows kept.')

In [ ]:
depts = gpd.read_file('departements-version-simplifiee.geojson')
_corsica = depts['code'].isin(['2A', '2B'])
depts = pd.concat([
    depts[~_corsica],
    gpd.GeoDataFrame({'code': ['20'], 'nom': ['Corse']},
                     geometry=[depts.loc[_corsica, 'geometry'].union_all()],
                     crs=depts.crs),
], ignore_index=True)
METRO = set(depts['code'])

dept_name = named.groupby(['dpt', 'preusuel'], as_index=False)['nombre'].sum()
dept_name = dept_name.merge(dept_total, on='dpt')
dept_name['per_1000'] = (1000 * dept_name['nombre'] / dept_name['dept_total']).round(3)
dept_name = dept_name.merge(depts[['code', 'nom']], left_on='dpt', right_on='code',
                            how='left').drop(columns=['code', 'dept_total'])

geo_features = alt.InlineData(values=json.loads(depts.to_json()),
                              format=alt.DataFormat(property='features'))
FRANCE_PROJECTION = dict(type='conicConformal', rotate=[-3, 0],
                         center=[0, 46.5], parallels=[44, 49])
print(f'{len(dept_name):,} (department, name) pairs embedded.')
dept_name.sample(3)

## Department x name x decade table *(from sketch 2.4)*

In [ ]:
# --- department x name x decade rate table (copied from sketch 2.4) so this
# --- notebook is self-contained; builds `dn_dec` (embedded inline below).
# (department, name, decade) rate table -> external JSON referenced by URL.
# ~745k rows: kept OUT of the spec (the json transformer would re-serialise it per
# layer); pandas writes it once and both layers point at the same file.
_slim4 = records.loc[records['decade'].notna(), ['dpt', 'decade', 'nombre']]
dec_tot4 = (_slim4.groupby(['dpt', 'decade'], as_index=False)['nombre'].sum()
            .rename(columns={'nombre': 'tot'}))
dn_dec = (named.dropna(subset=['decade'])
          .groupby(['dpt', 'preusuel', 'decade'], as_index=False)['nombre'].sum()
          .merge(dec_tot4, on=['dpt', 'decade']))
dn_dec['per_1000'] = (1000 * dn_dec['nombre'] / dn_dec['tot']).round(2)
dn_dec = dn_dec.drop(columns=['tot'])
dn_all = dept_name.copy()
dn_all['decade'] = 0
dn_dec = pd.concat([dn_dec[['dpt', 'preusuel', 'decade', 'nombre', 'per_1000']],
                    dn_all[['dpt', 'preusuel', 'decade', 'nombre', 'per_1000']]],
                   ignore_index=True)
dn_dec['decade'] = dn_dec['decade'].astype(int)
dn_dec = dn_dec.merge(depts[['code', 'nom']], left_on='dpt', right_on='code',
                      how='left').drop(columns='code')
print(f'{len(dn_dec):,} (dept, name, period) rows kept inline (no external file).')
del _slim4, dn_all

## The sketch

### Sketch 2.6 - Small-multiple maps, one per name (decade slider + map-count slider)

Choropleths with independent scales and a shared hover. Each map slot has its own name dropdown.
The decade slider lets you watch each name's stronghold move through the century,
and the Maps shown slider controls how many name dropdowns are visible and active, from 3 to 6.


In [ ]:
map_name_1 = alt.param(name='map_name_1', value='LUCIEN',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 01  '))
map_name_2 = alt.param(name='map_name_2', value='LEA',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 02  '))
map_name_3 = alt.param(name='map_name_3', value='GABRIEL',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 03  '))
map_name_4 = alt.param(name='map_name_4', value='NICOLAS',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 04  '))
map_name_5 = alt.param(name='map_name_5', value='EMMA',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 05  '))
map_name_6 = alt.param(name='map_name_6', value='LOUIS',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 06  '))
n_maps_26 = alt.param(name='n_maps_26', value=6,
                      bind=alt.binding_range(min=3, max=6, step=1, name='Maps shown  '))
decade_26 = alt.param(name='decade_26', value=2010,
                       bind=alt.binding_range(min=1900, max=2020, step=10,
                                              name='Decade  '))
mm_hover = alt.selection_point(fields=['nom'], on='pointerover', clear='pointerout', empty=False)

# The slider controls which map slots contain data; the executed notebook also
# hides dropdown controls above the selected count after papermill renders HTML.
def one_map(param_name, scheme, slot_label, slot_idx):
    filt = (f"n_maps_26 >= {slot_idx} && datum.preusuel == {param_name} "
            "&& datum.decade == decade_26 && isValid(datum.nom)")
    base = alt.Chart(dn_dec).transform_filter(filt).transform_lookup(
        lookup='dpt', from_=alt.LookupData(geo_features, key='properties.code', fields=['type', 'geometry'])
    )
    choro = base.mark_geoshape().encode(
        color=alt.Color('per_1000:Q', scale=alt.Scale(scheme=scheme),
                        legend=alt.Legend(title='/1000', orient='bottom', format='.1f')),
        stroke=alt.condition(mm_hover, alt.value('#111827'), alt.value('#cccccc')),
        strokeWidth=alt.condition(mm_hover, alt.value(1.8), alt.value(0.3)),
        tooltip=[alt.Tooltip('nom:N', title='Department'),
                 alt.Tooltip('preusuel:N', title='Name'),
                 alt.Tooltip('per_1000:Q', title='Births / 1000', format='.2f')],
    ).project(**FRANCE_PROJECTION)
    tag = alt.Chart(pd.DataFrame({'lon': [-4.6], 'lat': [51.3]})).transform_filter(
        f'n_maps_26 >= {slot_idx}'
    ).transform_calculate(
        label=param_name
    ).mark_text(align='left', fontSize=14, fontWeight='bold', color='#222').encode(
        longitude='lon:Q', latitude='lat:Q', text='label:N').project(**FRANCE_PROJECTION)
    return (choro + tag).properties(width=220, height=220, title=slot_label)

schemes = ['greens', 'reds', 'oranges', 'blues', 'purples', 'greys']
slot_defs = [
    ('map_name_1', schemes[0], 'Map 1', 1),
    ('map_name_2', schemes[1], 'Map 2', 2),
    ('map_name_3', schemes[2], 'Map 3', 3),
    ('map_name_4', schemes[3], 'Map 4', 4),
    ('map_name_5', schemes[4], 'Map 5', 5),
    ('map_name_6', schemes[5], 'Map 6', 6),
]
row_26_top = alt.hconcat(*[one_map(*slot_defs[i]) for i in range(3)]).resolve_scale(color='independent')
row_26_bottom = alt.hconcat(*[one_map(*slot_defs[i]) for i in range(3, 6)]).resolve_scale(color='independent')
sketch_2_6 = alt.vconcat(row_26_top, row_26_bottom).resolve_scale(color='independent').add_params(
    map_name_1, map_name_2, map_name_3, map_name_4, map_name_5, map_name_6,
    n_maps_26, mm_hover, decade_26,
).properties(
    title='2.6 - Regional rate of 3 to 6 selectable names (independent scales; slide decade and map count)')

# PNG snapshot with the default six names. The displayed chart above remains fully
# interactive, while the static export avoids URL fetch issues in vl-convert.
def one_map_png(nm, scheme):
    g = depts.merge(dn_dec[(dn_dec.preusuel == nm) & (dn_dec.decade == 2010)], how='right',
                    left_on='code', right_on='dpt').dropna(subset=['geometry'])
    return alt.Chart(g).mark_geoshape(stroke='#cccccc', strokeWidth=0.3).encode(
        color=alt.Color('per_1000:Q', scale=alt.Scale(scheme=scheme),
                        legend=alt.Legend(title='/1000', orient='bottom', format='.1f')),
        tooltip=[alt.Tooltip('nom:N', title='Department'),
                 alt.Tooltip('per_1000:Q', title='Births / 1000', format='.2f')],
    ).project(**FRANCE_PROJECTION).properties(width=220, height=220, title=nm)

row_26_png_top = alt.hconcat(
    one_map_png('LUCIEN', schemes[0]),
    one_map_png('LEA', schemes[1]),
    one_map_png('GABRIEL', schemes[2]),
).resolve_scale(color='independent')
row_26_png_bottom = alt.hconcat(
    one_map_png('NICOLAS', schemes[3]),
    one_map_png('EMMA', schemes[4]),
    one_map_png('LOUIS', schemes[5]),
).resolve_scale(color='independent')
sketch_2_6_png = alt.vconcat(row_26_png_top, row_26_png_bottom).resolve_scale(color='independent').properties(
    title='2.6 - Regional rate of six selectable names (2010 default PNG snapshot)')

sketch_2_6_png.save('sketch_2_6_small_multiples.png', ppi=120)
sketch_2_6
